# 04 · Export

把 `data/processed/village_to_nearest_library_{profile}.csv` 整理成終端使用者方便看的格式：

- `output/data/tainan_library_drive_time.csv` — 行車版 CSV
- `output/data/tainan_library_walk_time.csv` — 步行版 CSV
- `output/data/tainan_library_drive_time.xlsx` — Excel，三個 sheet：「行車時間」、「步行時間」、「圖書館清單」


In [2]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "output" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

libs = pd.read_csv(RAW_DIR / "tainan_libraries.csv")


def load(unit: str, profile: str):
    fp = PROC_DIR / f"{unit}_to_nearest_library_{profile}.csv"
    if not fp.exists():
        print(f"⚠️  {fp.name} not found")
        return None
    df = pd.read_csv(fp, dtype={"unit_id": str})
    sort_cols = ["district", "village_name"] if unit == "village" else ["unit_id"]
    sort_cols = [c for c in sort_cols if c in df.columns]
    return df.sort_values(sort_cols).reset_index(drop=True) if sort_cols else df


# Village-level (per-profile CSVs + combined Excel)
v_drive = load("village", "driving")
v_walk = load("village", "walking")

if v_drive is not None:
    p = OUT_DIR / "tainan_library_drive_time.csv"
    v_drive.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"✅ {p.name}: {len(v_drive)} rows (village)")
if v_walk is not None:
    p = OUT_DIR / "tainan_library_walk_time.csv"
    v_walk.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"✅ {p.name}: {len(v_walk)} rows (village)")

# Grid-level (per-profile CSVs, separate xlsx because much larger)
g_drive = load("grid", "driving")
g_walk = load("grid", "walking")

if g_drive is not None:
    p = OUT_DIR / "tainan_library_drive_time_grid.csv"
    g_drive.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"✅ {p.name}: {len(g_drive)} rows (grid)")
if g_walk is not None:
    p = OUT_DIR / "tainan_library_walk_time_grid.csv"
    g_walk.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"✅ {p.name}: {len(g_walk)} rows (grid)")

# Combined Excel (village data + library list; grid omitted to keep xlsx small)
xlsx = OUT_DIR / "tainan_library_drive_time.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    if v_drive is not None:
        v_drive.to_excel(w, sheet_name="行車時間", index=False)
    if v_walk is not None:
        v_walk.to_excel(w, sheet_name="步行時間", index=False)
    libs.to_excel(w, sheet_name="圖書館清單", index=False)
print(f"✅ Excel: {xlsx.name}")

(v_drive if v_drive is not None else v_walk).head()

✅ tainan_library_drive_time.csv: 650 rows (village)
✅ tainan_library_walk_time.csv: 650 rows (village)
⚠️  grid_to_nearest_library_walking.csv not found
✅ tainan_library_drive_time_grid.csv: 2412 rows (grid)
✅ Excel: tainan_library_drive_time.xlsx


,unit_id,centroid_lat,centroid_lon,nearest_library,library_lat,library_lon,distance_km,time_min,method,village_name,district
0,67000150010,23.129613,120.131759,七股區圖書館,23.140338,120.139167,1.7399,3.126667,osrm_driving,七股里,七股區
1,67000150022,23.113227,120.082433,七股區圖書館,23.140338,120.139167,8.0130,13.473333,osrm_driving,三股里,七股區
2,67000150017,23.157271,120.115806,七股區圖書館,23.140338,120.139167,4.7174,10.561667,osrm_driving,中寮里,七股區
3,67000150023,23.082966,120.064896,七股區圖書館,23.140338,120.139167,13.3041,18.693333,osrm_driving,十份里,七股區
4,67000150007,23.153206,120.082017,七股區圖書館,23.140338,120.139167,7.4032,9.755000,osrm_driving,塩埕里,七股區
